<a href="https://colab.research.google.com/github/Jasonmiller513/Dissertation/blob/Updates/WA_FINAL_SARSA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **STEP 1: CLEAN DATA**

# Import Extensions And Open File

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
import joblib
from collections import defaultdict
import re
df = pd.read_csv('DataCoSupplyChainDataset.csv', encoding='latin1')


# Remove Unnecessary Columns

In [3]:
# List of columns to remove
columns_to_remove = [

    # Identifier columns
    'Customer Id',
    'Category Id',
    'Department Id',

    # Personal/customer detail columns
    'Customer Email',
    'Customer Password',
    'Customer Fname',
    'Customer Lname',
    'Customer Street',
    'Customer Zipcode',
    'Customer Full Name',

    # High-cardinality text columns
    'Product Description',
    'Product Image',
    'Product Name',


    # Duplicate or Redundant Geographic Columns

'Customer Segment',
'Order City',
'Order State',]

# Remove only columns that actually exist in dataframe
existing_columns_to_remove = [
    col for col in columns_to_remove if col in df.columns]

# Drop columns
df = df.drop(columns=existing_columns_to_remove)

# Display remaining columns
print("Remaining Columns:")
print(df.columns.tolist())

# Display shape after removal
print("\nNew Dataset Shape:")
print(df.shape)

Remaining Columns:
['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Delivery Status', 'Late_delivery_risk', 'Category Name', 'Customer City', 'Customer Country', 'Customer State', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order Country', 'Order Customer Id', 'order date (DateOrders)', 'Order Id', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Region', 'Order Status', 'Order Zipcode', 'Product Card Id', 'Product Category Id', 'Product Price', 'Product Status', 'shipping date (DateOrders)', 'Shipping Mode']

New Dataset Shape:
(180519, 38)


# Identify missing or erroneous data


In [4]:

print(df.shape)
missing_values = df.isnull().sum()
print(missing_values[missing_values > 0])

(180519, 38)
Order Zipcode    155679
dtype: int64


In [5]:
duplicate_count = df.duplicated().sum()
print(f"Duplicate Rows: {duplicate_count}")

Duplicate Rows: 0


In [6]:
missing = df.isnull().sum()
print(missing[missing > 0])

Order Zipcode    155679
dtype: int64


In [7]:
num_cols = df.select_dtypes(include=['int64','float64']).columns

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())
    cat_cols = df.select_dtypes(include=['object']).columns

for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

In [8]:
non_negative_columns = [
    'Sales',
    'Order Item Quantity',
    'Order Item Total',
    'Product Price',
    'Days for shipping (scheduled)']

for col in non_negative_columns:

    if col in df.columns:

        # Count invalid negatives
        invalid_count = (df[col] < 0).sum()

        print(f"\n{col} Negative Values: {invalid_count}")

        # Replace negatives with median
        median_value = df[df[col] >= 0][col].median()

        df.loc[df[col] < 0, col] = median_value


Sales Negative Values: 0

Order Item Quantity Negative Values: 0

Order Item Total Negative Values: 0

Product Price Negative Values: 0


In [9]:
numeric_cols = df.select_dtypes(include=np.number).columns

outlier_summary = {}

for col in numeric_cols:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = ((df[col] < lower) | (df[col] > upper)).sum()

    outlier_summary[col] = outliers

outlier_df = pd.DataFrame.from_dict(
    outlier_summary,
    orient='index',
    columns=['Outlier Count']
)

print(outlier_df.sort_values('Outlier Count', ascending=False))

                               Outlier Count
Order Zipcode                          24818
Benefit per order                      18942
Order Profit Per Order                 18942
Order Item Profit Ratio                17300
Order Item Discount                     7537
Order Item Product Price                2048
Product Price                           2048
Sales per customer                      1943
Order Item Total                        1943
Longitude                               1414
Order Customer Id                       1198
Sales                                    488
Latitude                                   9
Late_delivery_risk                         0
Days for shipment (scheduled)              0
Days for shipping (real)                   0
Order Item Quantity                        0
Order Item Id                              0
Order Item Cardprod Id                     0
Order Item Discount Rate                   0
Order Id                                   0
Product Ca

# Winsorize Outliers

In [10]:
for col in numeric_cols:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[col] = np.where(df[col] < lower, lower, df[col])
    df[col] = np.where(df[col] > upper, upper, df[col])

# Log Transform Highly Skewed Variables

In [11]:
skew_cols = [
    'Order Item Total',
    'Order Profit Per Order',
    'Shipping Cost',
    'Profit Per Unit'
]

for col in skew_cols:
    if col in df.columns:
        df[col] = np.log1p(df[col] - df[col].min())

In [12]:
print("\nRemaining Missing Values:")
print(df.isnull().sum().sum())

print("\nFinal Dataset Shape:")
print(df.shape)

print("\nDataset Preview:")
print(df.head())


Remaining Missing Values:
0

Final Dataset Shape:
(180519, 38)

Dataset Preview:
       Type  Days for shipping (real)  Days for shipment (scheduled)  \
0     DEBIT                       3.0                            4.0   
1  TRANSFER                       5.0                            4.0   
2      CASH                       4.0                            4.0   
3     DEBIT                       3.0                            4.0   
4   PAYMENT                       2.0                            4.0   

   Benefit per order  Sales per customer   Delivery Status  \
0          91.250000          314.640015  Advance shipping   
1         -79.700005          311.359985     Late delivery   
2         -79.700005          309.720001  Shipping on time   
3          22.860001          304.809998  Advance shipping   
4         134.210007          298.250000  Advance shipping   

   Late_delivery_risk   Category Name Customer City Customer Country  ...  \
0                 0.0  Sporting Goo

# Remove Constant and Near-Constant Features

In [13]:
constant_cols = [
    col for col in df.columns
    if df[col].nunique() == 1
]

print("Constant Features:")
print(constant_cols)

df.drop(columns=constant_cols, inplace=True)
near_constant_cols = []

for col in df.columns:

    freq = df[col].value_counts(normalize=True)

    # Check if freq is empty (e.g., column was entirely NaN) or if a single value dominates
    if freq.empty or freq.iloc[0] > 0.99:
        near_constant_cols.append(col)

print("Near Constant Features:")
print(near_constant_cols)

df.drop(columns=near_constant_cols, inplace=True)

Constant Features:
['Order Zipcode', 'Product Status']
Near Constant Features:
[]


# Remove Duplicate Rows

In [14]:
print("Rows before:", len(df))

df.drop_duplicates(inplace=True)

print("Rows after:", len(df))

Rows before: 180519
Rows after: 180519


# Verify Dataset Quality

In [15]:
print("Remaining Missing Values:")
print(df.isnull().sum().sum())

print("Dataset Shape:")
print(df.shape)

Remaining Missing Values:
0
Dataset Shape:
(180519, 36)


# Recompute Numeric Columns

In [16]:
numeric_cols = df.select_dtypes(include=['int64','float64']).columns.tolist()

print(f"Number of numeric features: {len(numeric_cols)}")
print(numeric_cols)

Number of numeric features: 22
['Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Late_delivery_risk', 'Latitude', 'Longitude', 'Order Customer Id', 'Order Id', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Product Card Id', 'Product Category Id', 'Product Price']


#Save Data

In [17]:
# Save cleaned RL dataset to CSV in Google Colab

output_file = 'dataco_rl_cleaned.csv'

# Save dataframe
df.to_csv(output_file, index=False, encoding="latin1")

print(f"Dataset saved as: {output_file}")

Dataset saved as: dataco_rl_cleaned.csv


# **STEP 2: FEATURE ENGINEERING**

In [18]:
df = pd.read_csv('dataco_rl_cleaned.csv', encoding='latin1')


print("Dataset Loaded")
print(df.shape)

Dataset Loaded
(180519, 36)


# Date and Time Features

In [19]:
# Convert order date to datetime
df['order date (DateOrders)'] = pd.to_datetime(
    df['order date (DateOrders)'],
    errors='coerce')

# Extract temporal features
df['order_year'] = df['order date (DateOrders)'].dt.year
df['order_month'] = df['order date (DateOrders)'].dt.month
df['order_day'] = df['order date (DateOrders)'].dt.day
df['order_dayofweek'] = df['order date (DateOrders)'].dt.dayofweek
df['is_weekend'] = df['order_dayofweek'].isin([5,6]).astype(int)


# Create a Product-Region Matrix and Network

In [20]:
# Create a Product-Region Matrix
product_region = (
    df.groupby("Product Category Id")["Order Region"]
      .nunique()
      .reset_index(name="num_regions")
      .sort_values("num_regions", ascending=False))

In [21]:
order_features = (
    df.groupby("Order Id")
      .agg(
          unique_products=("Product Category Id", "nunique"),
          total_quantity=("Order Item Quantity", "sum"),
          total_sales=("Sales", "sum"))
      .reset_index())

order_features["complex_order"] = (
    order_features["unique_products"] >= 3
).astype(int)

In [22]:
network = (
    df.groupby(["Product Category Id", "Order Region"])
      .agg(
          orders=("Order Id", "nunique"),
          quantity=("Order Item Quantity", "sum"),
          revenue=("Sales", "sum"))
      .reset_index())

# Explore Product Statistics

In [23]:
product_stats = (
    network.groupby("Product Category Id")
           .agg(
               regions_served=("Order Region", "nunique"),
               total_orders=("orders", "sum"),
               total_quantity=("quantity", "sum"),
               total_revenue=("revenue", "sum"))
           .reset_index())


# Create Shipping Mode Statistics

In [24]:
import pandas as pd
import numpy as np

# Ensure 'Shipping Mode_str' and 'Order Region_str' are available
df['Shipping Mode_str'] = df['Shipping Mode']
df['Order Region_str'] = df['Order Region']


# Historical shipping mode performance
mode_stats = (
    df.groupby('Shipping Mode_str')
      .agg({
          'Order Profit Per Order':'mean',
          'Late_delivery_risk':'mean',
          'Days for shipping (real)':'mean'
      })
      .reset_index()
)

print(mode_stats)

  Shipping Mode_str  Order Profit Per Order  Late_delivery_risk  \
0       First Class                4.364252            0.953225   
1          Same Day                4.337734            0.457430   
2      Second Class                4.355014            0.766328   
3    Standard Class                4.354793            0.380717   

   Days for shipping (real)  
0                  2.000000  
1                  0.478279  
2                  3.990828  
3                  3.995907  


# Define Multi-Region Stock

In [25]:
region_threshold = product_stats["regions_served"].median()
order_threshold = product_stats["total_orders"].median()

product_stats["likely_multi_region_stocked"] = (
    (product_stats["regions_served"] >= region_threshold)
    &
    (product_stats["total_orders"] >= order_threshold)
).astype(int)

#Create a stock score

In [26]:
product_stats["stocking_score"] = (
    0.5 *
    (
        product_stats["regions_served"]
        / product_stats["regions_served"].max())
    +
    0.5 *
    (
        product_stats["total_orders"]
        / product_stats["total_orders"].max()))


# Merge the network

In [27]:
network = network.merge(
    product_stats[
        [
            "Product Category Id",
            "regions_served",
            "stocking_score",
            "likely_multi_region_stocked"
        ]
    ],
    on="Product Category Id",
    how="left")

# Find Edge Weight

In [28]:
network["edge_weight"] = (
    0.4 *
    (network["orders"] / network["orders"].max())
    +
    0.3 *
    (network["quantity"] / network["quantity"].max())
    +
    0.3 *
    (network["revenue"] / network["revenue"].max()))


 # Find candidate shipping regions

In [29]:
candidate_regions = (
    network.sort_values(
        ["Product Category Id", "edge_weight"],
        ascending=[True, False])
    .groupby("Product Category Id")
    .head(5))

# Save Outputs

In [30]:
network.to_csv(
    "product_region_network.csv",
    index=False
)

candidate_regions.to_csv(
    "candidate_fulfillment_regions.csv",
    index=False
)

product_stats.to_csv(
    "product_stocking_scores.csv",
    index=False
)

print("Files created successfully")

Files created successfully


# Shipping Engineering

In [31]:
df['order date (DateOrders)'] = pd.to_datetime(
    df['order date (DateOrders)'],
    errors='coerce'
)
df['shipping date (DateOrders)'] = pd.to_datetime(
    df['shipping date (DateOrders)'],
    errors='coerce'
)

# Actual shipping time

df['actual_ship_days'] = (
    df['shipping date (DateOrders)']
    - df['order date (DateOrders)']
).dt.days

# Difference from scheduled

df['shipping_delay'] = (
    df['Days for shipping (real)']
    - df['Days for shipment (scheduled)']
)

# Binary on-time flag

df['on_time_delivery'] = (
    df['shipping_delay'] <= 0
).astype(int)

# Late shipment flag

df['late_delivery'] = (
    df['shipping_delay'] > 0
).astype(int)

# Profit Engineering

In [32]:
# Margin percentage

df['margin_pct'] = (
    df['Order Profit Per Order']
    / df['Sales']
)

# Profit per item

df['profit_per_item'] = (
    df['Order Profit Per Order']
    / df['Order Item Quantity']
)

# Discount percentage

df['discount_pct'] = (
    df['Order Item Discount']
    / df['Order Item Product Price']
)

# Product Demand Features

In [33]:
product_stats = (
    df.groupby('Product Category Id')
      .agg(
          product_orders=('Order Id','nunique'),
          product_quantity=('Order Item Quantity','sum'),
          product_revenue=('Sales','sum')
      )
      .reset_index()
)

df = df.merge(
    product_stats,
    on='Product Category Id',
    how='left'
)

# Regional Features

In [34]:
region_stats = (
    df.groupby('Order Region')
      .agg(
          region_orders=('Order Id','nunique'),
          region_sales=('Sales','sum'),
          region_profit=('Order Profit Per Order','sum')
      )
      .reset_index()
)

df = df.merge(
    region_stats,
    on='Order Region',
    how='left'
)

# Fulfillment Network Features

In [35]:
stocking = pd.read_csv(
    "product_stocking_scores.csv", encoding='latin1'
)

df = df.merge(
    stocking[
        [
            'Product Category Id',
            'stocking_score',
            'regions_served',
            'likely_multi_region_stocked'
        ]
    ],
    on='Product Category Id',
    how='left'
)

# Shipping Mode Features

In [36]:
speed_mapping = {
    'Same Day':4,
    'First Class':3,
    'Second Class':2,
    'Standard Class':1
}
df['shipping_mode_speed'] = df['Shipping Mode'].map(speed_mapping)

# Create string versions of Shipping Mode and Order Region for later use
df['Shipping Mode_str'] = df['Shipping Mode']
df['Order Region_str'] = df['Order Region']

action_id_mapping = {
    'Standard Class': 0,    # Maps to key 0 in shipping_cost_map
    'Second Class': 1,     # Maps to key 1 in shipping_cost_map
    'First Class': 2,      # Maps to key 2 in shipping_cost_map
    'Same Day': 3          # Maps to key 3 in shipping_cost_map
}
df['action_id'] = df['Shipping Mode'].map(action_id_mapping)

shipping_cost_map = {
    0: 1.0,    # Standard
    1: 1.5,    # Second
    2: 2.0,    # First
    3: 3.0     # Same Day
}

BASE_COST = 10

df['Mode_Cost'] = (
    BASE_COST *
    df['action_id'].map(shipping_cost_map)
)

# Order Complexity Features

In [37]:
order_complexity = (
    df.groupby('Order Id')
      .agg(
          unique_products=('Product Category Id','nunique'),
          total_quantity=('Order Item Quantity','sum'),
          total_sales=('Sales','sum')
      )
      .reset_index()
)

df = df.merge(
    order_complexity,
    on='Order Id',
    how='left'
)

df['complex_order'] = (
    df['unique_products'] >= 3
).astype(int)

# Route-Based Shipping Costs

In [38]:
df = df.copy()

df["Shipping Cost"] = df["Mode_Cost"]

df["Route"] = (
    df["Order Region"].astype(str)
    + " -> "
    + df["Order Country"].astype(str)
)

route_mode_cost = (
    df.groupby(["Route", "Shipping Mode"])["Shipping Cost"]
      .mean()
      .reset_index()
      .rename(columns={"Shipping Cost": "Route_Mode_Cost"})
)

df = df.merge(
    route_mode_cost,
    on=["Route", "Shipping Mode"],
    how="left"
)

# RL State Features

In [39]:
state_features = [
    'Product Category Id',
    'stocking_score',
    'margin_pct',
    'shipping_delay',
    'regions_served',
    'profit_per_item'
]

# Reward Features (69)

In [40]:
df["reward"] = (
    7 * df["Order Profit Per Order"]
    - 3.0 * df["Days for shipping (real)"]
    - 6.0 * df["Late_delivery_risk"]
    - 0.10 * df["Route_Mode_Cost"]
)

# Save Engineered Dataset

In [41]:
df.to_csv(
    "dataco_rl_feature_engineered.csv",
    index=False, encoding="latin1"
)

print(df.shape)

(180519, 70)


# **STEP 3: ENCODING CATEGORICAL VARIABLES**

In [42]:

df = pd.read_csv('dataco_rl_feature_engineered.csv', encoding='latin1')

print("Dataset Loaded")
print(df.shape)

Dataset Loaded
(180519, 70)


In [43]:
one_hot_cols = [
    'Shipping Mode',
    'Market',
    'Order Region'
]

df = pd.get_dummies(
    df,
    columns=one_hot_cols,
    drop_first=False
)

df.to_csv(
    "dataco_rl_onehot.csv",
    index=False, encoding="latin1"
)

print(df.shape)

(180519, 99)


# **STEP 4: TRAIN/TEST TEMPORAL SPLIT**

In [44]:
import pandas as pd

df = pd.read_csv("dataco_rl_onehot.csv", encoding='latin1')


df['order date (DateOrders)'] = pd.to_datetime(
    df['order date (DateOrders)']
)

df = df.sort_values(
    'order date (DateOrders)'
)

n = len(df)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = df.iloc[:train_end]
val_df = df.iloc[train_end:val_end]
test_df = df.iloc[val_end:]

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(126363, 99)
(27078, 99)
(27078, 99)


In [45]:
# Save datasets

train_df.to_csv(
    "dataco_rl_train.csv",
    index=False, encoding="latin1"
)

val_df.to_csv(
    "dataco_rl_validation.csv",
    index=False, encoding="latin1"
)

test_df.to_csv(
    "dataco_rl_test.csv",
    index=False, encoding="latin1"
)

print("Files saved successfully")

Files saved successfully


# STEP 5: STATE-SPACE DISCRETIZATION

# Feature Selection

In [46]:
state_features = [
    'Profit Margin',
    'Shipping Cost Ratio',
    'Historical On-Time Rate',
    'Regional Delay Score',
    'Product Category Id'
]

# Fit Bins Using Training Data Only

In [47]:
train_df = pd.read_csv("dataco_rl_train.csv", encoding='latin1')

# Use train_df instead of df for binning operations
train_df['shipping_cost_bin'] = pd.qcut(
    train_df['Shipping Cost'],
    q=5,
    labels=False,
    duplicates='drop'
)

train_df['order_value_bin'] = pd.qcut(
    train_df['Order Item Total'],
    q=5,
    labels=False,
    duplicates='drop'
)


train_df['Order Region_bin'] = 0

exclude_cols = [
    'Reward',
    'Late_delivery_risk',
    'Action_ID'
]

continuous_features = [
    col for col in train_df.select_dtypes(include='number').columns
    if col not in exclude_cols
    and not col.endswith('_bin') # Exclude the newly created bin columns from being binned again
]

bin_edges = {}

# Create 10 bins for each feature
for col in continuous_features:
    # Handle cases where all values are the same to prevent qcut errors
    if train_df[col].nunique() == 1:
        # If all values are the same, create a single bin from min to min (or min to min + epsilon)
        min_val = train_df[col].min()
        bins = np.array([min_val, min_val + 1e-9]) # Add a small epsilon to ensure two distinct points
    else:
        _, bins = pd.qcut(
            train_df[col],
            q=10,
            labels=False,
            retbins=True,
            duplicates='drop'
        )
    bin_edges[col] = bins

# Save bin definitions
joblib.dump(
    bin_edges,
    "state_bins.pkl"
)

print("Bins created successfully")

Bins created successfully


In [48]:
bin_counts = {}

for col in continuous_features:
    _, bins = pd.qcut(
        train_df[col],
        q=10,
        retbins=True,
        duplicates='drop'
    )

    bin_counts[col] = len(bins) - 1

print(bin_counts)

{'Days for shipping (real)': 5, 'Days for shipment (scheduled)': 3, 'Benefit per order': 10, 'Sales per customer': 10, 'Latitude': 10, 'Longitude': 10, 'Order Customer Id': 10, 'Order Id': 10, 'Order Item Cardprod Id': 8, 'Order Item Discount': 10, 'Order Item Discount Rate': 10, 'Order Item Id': 10, 'Order Item Product Price': 8, 'Order Item Profit Ratio': 10, 'Order Item Quantity': 4, 'Sales': 9, 'Order Item Total': 10, 'Order Profit Per Order': 10, 'Product Card Id': 8, 'Product Category Id': 8, 'Product Price': 8, 'order_year': 2, 'order_month': 10, 'order_day': 10, 'order_dayofweek': 6, 'is_weekend': 1, 'actual_ship_days': 5, 'shipping_delay': 5, 'on_time_delivery': 1, 'late_delivery': 1, 'margin_pct': 10, 'profit_per_item': 10, 'discount_pct': 10, 'product_orders': 8, 'product_quantity': 9, 'product_revenue': 8, 'region_orders': 9, 'region_sales': 9, 'region_profit': 9, 'stocking_score': 8, 'regions_served': 1, 'likely_multi_region_stocked': 1, 'shipping_mode_speed': 3, 'action_i

# Apply Bins to Train/Test Data

In [49]:
train_df = pd.read_csv("dataco_rl_train.csv", encoding='latin1')
test_df = pd.read_csv("dataco_rl_test.csv", encoding='latin1')

bin_edges = joblib.load("state_bins.pkl")

continuous_features = list(bin_edges.keys())

for col in continuous_features:

    train_df[col + "_bin"] = np.digitize(
        train_df[col],
        bin_edges[col][1:-1]
    )

    test_df[col + "_bin"] = np.digitize(
        test_df[col],
        bin_edges[col][1:-1]
    )

train_df.to_csv(
    "dataco_rl_train_discrete.csv",
    index=False, encoding="latin1"
)

test_df.to_csv(
    "dataco_rl_test_discrete.csv",
    index=False, encoding="latin1"
)

print("Discretization complete")

Discretization complete


# Build Action Lookup Table

In [51]:
from sklearn.preprocessing import LabelEncoder
import joblib
import pandas as pd
import numpy as np


# Load split files

train_df = pd.read_csv("dataco_rl_train_discrete.csv", encoding='latin1')

val_df = pd.read_csv("dataco_rl_validation.csv", encoding='latin1')
test_df = pd.read_csv("dataco_rl_test.csv", encoding='latin1')


# Load candidate fulfillment regions

candidate_regions = pd.read_csv("candidate_fulfillment_regions.csv", encoding='latin1')

# Rename fulfillment region column if needed
if "Order Region" in candidate_regions.columns:
    candidate_regions = candidate_regions.rename(
        columns={"Order Region": "Fulfillment_Region"}
    )

# Keep only needed columns
candidate_regions = candidate_regions[
    ["Product Category Id", "Fulfillment_Region"]
].drop_duplicates()


# Merge fulfillment regions into train/val/test

train_df = train_df.merge(
    candidate_regions,
    on="Product Category Id",
    how="left"
)

val_df = val_df.merge(
    candidate_regions,
    on="Product Category Id",
    how="left"
)

test_df = test_df.merge(
    candidate_regions,
    on="Product Category Id",
    how="left"
)


# Restore Shipping Mode Text
shipping_mode_map_reverse = {
    1: "Standard Class",
    2: "Second Class",
    3: "First Class",
    4: "Same Day"
}

for df_iter in [train_df, val_df, test_df]:
    if "Shipping Mode_str" not in df_iter.columns:
        df_iter["Shipping Mode_str"] = df_iter["shipping_mode_speed"].map(
            shipping_mode_map_reverse
        )




# Re-apply binning to val_df and test_df after loading non-discrete versions

bin_edges = joblib.load("state_bins.pkl")
continuous_features = list(bin_edges.keys())

# Ensure val_df and test_df are updated in place with binned columns
for df_obj in [val_df, test_df]:
    for col in continuous_features:
        if col in df_obj.columns:
            if len(bin_edges[col]) > 2:
                df_obj.loc[:, col + "_bin"] = np.digitize(
                    df_obj[col],
                    bin_edges[col][1:-1]
                )
            else:
                df_obj.loc[:, col + "_bin"] = 0



# Create route for all dataframes
train_df["Route"] = (
    train_df["Order Region_str"].astype(str)
    + " -> "
    + train_df["Order Country"].astype(str)
)
val_df["Route"] = (
    val_df["Order Region_str"].astype(str)
    + " -> "
    + val_df["Order Country"].astype(str)
)
test_df["Route"] = (
    test_df["Order Region_str"].astype(str)
    + " -> "
    + test_df["Order Country"].astype(str)
)

# Calculate route statistics using only train_df
# The 'Route' column already encapsulates 'Order Region_str' and 'Order Country'.
route_stats = train_df.groupby(['Route']).agg(
    route_profit_avg=('Order Profit Per Order', 'mean'),
    route_days_avg=('Days for shipping (real)', 'mean'),
    route_late_risk_avg=('Late_delivery_risk', 'mean')
).reset_index()

# Merge route statistics into all dataframes
train_df = train_df.merge(
    route_stats,
    on=['Route'],
    how='left'
)

val_df = val_df.merge(
    route_stats,
    on=['Route'],
    how='left'
)

test_df = test_df.merge(
    route_stats,
    on=['Route'],
    how='left'
)


# Create combined action
# Fulfillment Region + Route + Shipping Mode
train_df["action"] = (
    train_df["Fulfillment_Region"].astype(str)
    + " | "
    + train_df["Route"].astype(str)
    + " | "
    + train_df["Shipping Mode_str"].astype(str)
)
val_df["action"] = (
    val_df["Fulfillment_Region"].astype(str)
    + " | "
    + val_df["Route"].astype(str)
    + " | "
    + val_df["Shipping Mode_str"].astype(str)
)
test_df["action"] = (
    test_df["Fulfillment_Region"].astype(str)
    + " | "
    + test_df["Route"].astype(str)
    + " | "
    + test_df["Shipping Mode_str"].astype(str)
)


# Create state IDs

state_cols = [
    "Product Category Id_bin",
    "stocking_score_bin",
    "margin_pct_bin",
    "shipping_delay_bin",
    "Order Region_bin",
    "Shipping Cost_bin",
    "Order Item Total_bin"
]

# Ensure 'Order Region_bin' is present in all dataframes
for df_obj in [train_df, val_df, test_df]:
    if 'Order Region_bin' not in df_obj.columns:
        df_obj['Order Region_bin'] = 0

# Check for missing state columns in train_df first
missing_train_state_cols = [col for col in state_cols if col not in train_df.columns]
if missing_train_state_cols:
    raise ValueError(f"Missing state columns in train_df: {missing_train_state_cols}")

train_df["state"] = train_df[state_cols].astype(str).agg("|".join, axis=1)

state_encoder = LabelEncoder()
train_df["state_id"] = state_encoder.fit_transform(train_df["state"])


# Create action IDs

action_encoder = LabelEncoder()
train_df["action_id"] = action_encoder.fit_transform(train_df["action"])

# Handle unseen actions/states in validation/test
valid_actions = set(action_encoder.classes_)
valid_states = set(state_encoder.classes_)

# Process val_df and test_df for state and action IDs
# This loop was problematic for in-place modification; now using direct assignment after filtering

# Re-process val_df and test_df correctly to update external variables
val_df["state"] = val_df[state_cols].astype(str).agg("|".join, axis=1)
val_df = val_df[val_df["action"].isin(valid_actions)].copy()
val_df = val_df[val_df["state"].isin(valid_states)].copy()
val_df["action_id"] = action_encoder.transform(val_df["action"])
val_df["state_id"] = state_encoder.transform(val_df["state"])

test_df["state"] = test_df[state_cols].astype(str).agg("|".join, axis=1)
test_df = test_df[test_df["action"].isin(valid_actions)].copy()
test_df = test_df[test_df["state"].isin(valid_states)].copy()
test_df["action_id"] = action_encoder.transform(test_df["action"])
test_df["state_id"] = state_encoder.transform(test_df["state"])


# Save encoders

joblib.dump(state_encoder, "state_encoder.pkl")
joblib.dump(action_encoder, "action_encoder.pkl")


# Final checks

n_states = train_df["state_id"].nunique()
n_actions = train_df["action_id"].nunique()

print("Rows:", len(train_df))
print("States:", n_states)
print("Actions:", n_actions)

print("\nSample actions:")
print(train_df["action"].head(10))

print("\nAction encoder classes:")
print(action_encoder.classes_[:10])

Rows: 631815
States: 3564
Actions: 4633

Sample actions:
0    Central America | Central America -> México | ...
1    Western Europe | Central America -> México | S...
2    South America | Central America -> México | St...
3    Northern Europe | Central America -> México | ...
4    Oceania | Central America -> México | Standard...
5    Central America | South America -> Colombia | ...
6    Western Europe | South America -> Colombia | S...
7    South America | South America -> Colombia | St...
8    Southern Europe | South America -> Colombia | ...
9    Northern Europe | South America -> Colombia | ...
Name: action, dtype: object

Action encoder classes:
['Caribbean | Canada -> Canada | First Class'
 'Caribbean | Canada -> Canada | Same Day'
 'Caribbean | Canada -> Canada | Second Class'
 'Caribbean | Canada -> Canada | Standard Class'
 'Caribbean | Caribbean -> Barbados | First Class'
 'Caribbean | Caribbean -> Barbados | Second Class'
 'Caribbean | Caribbean -> Barbados | Standard Class

In [52]:
print("Rows:", len(train_df))
print("States:", train_df["state_id"].nunique())
print("Actions:", train_df["action_id"].nunique())
print(train_df["action"].head())

Rows: 631815
States: 3564
Actions: 4633
0    Central America | Central America -> México | ...
1    Western Europe | Central America -> México | S...
2    South America | Central America -> México | St...
3    Northern Europe | Central America -> México | ...
4    Oceania | Central America -> México | Standard...
Name: action, dtype: object


In [53]:
print("Q shape should be:")
print("States:", train_df["state_id"].nunique())
print("Actions:", train_df["action_id"].nunique())

n_states = train_df["state_id"].nunique()
n_actions = train_df["action_id"].nunique()

Q = np.zeros((n_states, n_actions))

print(Q.shape)

Q shape should be:
States: 3564
Actions: 4633
(3564, 4633)


# Check State Space Size

In [54]:
print("States:", train_df["state_id"].nunique())
print("Actions:", train_df["action_id"].nunique())

States: 3564
Actions: 4633


In [55]:
print(
    "Unique states:",
    train_df['state'].nunique()
)

print(
    "Unique actions:",
    train_df['action'].nunique()
)

print(
    "Q-table size:",
    train_df['state'].nunique()
    *
    train_df['action'].nunique()
)

Unique states: 3564
Unique actions: 4633
Q-table size: 16512012


# **STEP 7: SARSA IMPLEMENTATION**


In [59]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import joblib
import time

# 1. Precompute arrays
# -----------------------------

states = train_df["sarsa_state_id"].astype(int).to_numpy()
actions_observed = train_df["sarsa_action_id"].astype(int).to_numpy()
rewards = train_df["sarsa_reward"].astype(float).to_numpy()

current_states = states[:-1]
next_states = states[1:]

n_steps = len(current_states)

print("Training transitions:", n_steps)

# -----------------------------
# 2. Precompute valid actions by state
# -----------------------------

valid_actions_array = {}

for s in range(n_states_sarsa):
    acts = train_df.loc[
        train_df["sarsa_state_id"] == s,
        "sarsa_action_id"
    ].unique()

    valid_actions_array[s] = np.array(acts, dtype=int)

# -----------------------------
# 3. Faster helper functions
# -----------------------------

def fast_choose_action(state, epsilon):
    valid_actions = valid_actions_array.get(state)

    if valid_actions is None or len(valid_actions) == 0:
        return None

    if np.random.random() < epsilon:
        return np.random.choice(valid_actions)

    q_vals = Q_sarsa[state, valid_actions]
    return valid_actions[np.argmax(q_vals)]


def fast_expected_q(next_state, epsilon):
    valid_actions = valid_actions_array.get(next_state)

    if valid_actions is None or len(valid_actions) == 0:
        return 0.0

    q_vals = Q_sarsa[next_state, valid_actions]

    best_idx = np.argmax(q_vals)

    n_valid = len(valid_actions)

    probs = np.full(n_valid, epsilon / n_valid)
    probs[best_idx] += 1.0 - epsilon

    return np.dot(probs, q_vals)

# -----------------------------
# 4. Faster training parameters
# -----------------------------

alpha = 0.15
alpha_decay = 0.9995
alpha_min = 0.03

gamma = 0.75

epsilon = 1.0
epsilon_decay = 0.997
epsilon_min = 0.03

episodes = 500   # start lower for full dataset

# -----------------------------
# 5. Fast Expected SARSA loop
# -----------------------------

start_time = time.time()

previous_policy = np.full(n_states_sarsa, -1)

for episode in range(episodes):
    total_td_error = 0.0
    updates = 0

    for i in range(n_steps):
        state = current_states[i]
        next_state = next_states[i]

        action = fast_choose_action(state, epsilon)

        if action is None:
            continue

        reward = rewards[i]
        expected_next_q = fast_expected_q(next_state, epsilon)

        old_q = Q_sarsa[state, action]

        target = reward + gamma * expected_next_q
        td_error = target - old_q

        Q_sarsa[state, action] += alpha * td_error

        total_td_error += abs(td_error)
        updates += 1

    epsilon = max(epsilon_min, epsilon * epsilon_decay)
    alpha = max(alpha_min, alpha * alpha_decay)

    # Current greedy policy
    current_policy = np.full(n_states_sarsa, -1)

    for s in range(n_states_sarsa):
        valid_actions = valid_actions_array.get(s)

        if valid_actions is not None and len(valid_actions) > 0:
            current_policy[s] = valid_actions[
                np.argmax(Q_sarsa[s, valid_actions])
            ]

    if episode == 0:
        policy_change_pct = 100.0
    else:
        policy_change_pct = (
            np.mean(current_policy != previous_policy) * 100
        )

    previous_policy = current_policy.copy()

    avg_td_error = total_td_error / max(updates, 1)

    if episode % 25 == 0:
        elapsed = time.time() - start_time
        print(
            f"Episode {episode}: "
            f"Avg TD Error = {avg_td_error:.6f}, "
            f"Policy Change = {policy_change_pct:.2f}%, "
            f"Epsilon = {epsilon:.4f}, "
            f"Alpha = {alpha:.4f}, "
            f"Elapsed = {elapsed/60:.2f} min"
        )

print("Fast Expected SARSA training complete.")
print(f"Total runtime: {(time.time() - start_time)/60:.2f} minutes")

Training transitions: 631814
Episode 0: Avg TD Error = 0.883217, Policy Change = 100.00%, Epsilon = 0.9970, Alpha = 0.1499, Elapsed = 0.41 min
Episode 25: Avg TD Error = 0.884599, Policy Change = 67.06%, Epsilon = 0.9249, Alpha = 0.1481, Elapsed = 10.53 min
Episode 50: Avg TD Error = 0.883413, Policy Change = 70.65%, Epsilon = 0.8579, Alpha = 0.1462, Elapsed = 20.45 min
Episode 75: Avg TD Error = 0.882632, Policy Change = 71.58%, Epsilon = 0.7959, Alpha = 0.1444, Elapsed = 30.17 min
Episode 100: Avg TD Error = 0.881392, Policy Change = 74.69%, Epsilon = 0.7383, Alpha = 0.1426, Elapsed = 39.71 min
Episode 125: Avg TD Error = 0.881209, Policy Change = 73.29%, Epsilon = 0.6848, Alpha = 0.1408, Elapsed = 49.08 min
Episode 150: Avg TD Error = 0.879631, Policy Change = 75.06%, Epsilon = 0.6353, Alpha = 0.1391, Elapsed = 58.21 min


KeyboardInterrupt: 

# Extract Policy

In [ ]:
sarsa_policy_action_ids = []

for s in range(n_states_sarsa):
    valid_actions = valid_state_actions_sarsa.get(s, [])

    if len(valid_actions) == 0:
        best_action = np.argmax(Q_sarsa[s])
    else:
        best_action = max(valid_actions, key=lambda a: Q_sarsa[s, a])

    sarsa_policy_action_ids.append(best_action)

sarsa_policy_df = pd.DataFrame({
    "sarsa_state_id": np.arange(n_states_sarsa),
    "sarsa_action_id": sarsa_policy_action_ids,
    "recommended_action": action_encoder_sarsa.inverse_transform(sarsa_policy_action_ids)
})

split_cols = sarsa_policy_df["recommended_action"].str.split(" \\| ", expand=True)

if split_cols.shape[1] == 3:
    sarsa_policy_df["Recommended_Fulfillment_Region"] = split_cols[0]
    sarsa_policy_df["Recommended_Route"] = split_cols[1]
    sarsa_policy_df["Recommended_Shipping_Mode"] = split_cols[2]
elif split_cols.shape[1] == 2:
    sarsa_policy_df["Recommended_Route"] = split_cols[0]
    sarsa_policy_df["Recommended_Shipping_Mode"] = split_cols[1]
else:
    sarsa_policy_df["Recommended_Shipping_Mode"] = sarsa_policy_df["recommended_action"]

display(sarsa_policy_df.head())

#Evaluate Outcomes

In [ ]:
sarsa_eval_df = train_df.merge(
    sarsa_policy_df,
    on="sarsa_state_id",
    how="left"
)

action_outcomes_sarsa = train_df.groupby("action")[[
    "Order Profit Per Order",
    "Days for shipping (real)",
    "Late_delivery_risk"
]].mean()

sarsa_eval_df = sarsa_eval_df.merge(
    action_outcomes_sarsa,
    left_on="recommended_action",
    right_index=True,
    suffixes=("_historical", "_policy")
)

historical_profit = sarsa_eval_df["Order Profit Per Order_historical"].mean()
policy_profit = sarsa_eval_df["Order Profit Per Order_policy"].mean()

historical_days = sarsa_eval_df["Days for shipping (real)_historical"].mean()
policy_days = sarsa_eval_df["Days for shipping (real)_policy"].mean()

historical_late_risk = sarsa_eval_df["Late_delivery_risk_historical"].mean()
policy_late_risk = sarsa_eval_df["Late_delivery_risk_policy"].mean()

profit_change_pct = ((policy_profit - historical_profit) / historical_profit) * 100
days_change_pct = ((historical_days - policy_days) / historical_days) * 100
late_risk_change_pct = ((historical_late_risk - policy_late_risk) / historical_late_risk) * 100

print(f"Historical Profit: {historical_profit:.4f}")
print(f"SARSA Policy Profit: {policy_profit:.4f}")
print(f"Profit Change %: {profit_change_pct:.2f}%")

print(f"Historical Days: {historical_days:.4f}")
print(f"SARSA Policy Days: {policy_days:.4f}")
print(f"Delivery Time Reduction %: {days_change_pct:.2f}%")

print(f"Historical Late Risk: {historical_late_risk:.4f}")
print(f"SARSA Policy Late Risk: {policy_late_risk:.4f}")
print(f"Late Risk Reduction %: {late_risk_change_pct:.2f}%")

if profit_change_pct > 0 and days_change_pct > 0 and late_risk_change_pct > 0:
    print("SARSA goal achieved")
else:
    print("SARSA goal not achieved")